# Task 9 — Three Validation Experiments

Standalone companion notebook to Tasks 1–8. This executes three validations: randomized sensor layouts, separate spatial/temporal orders, and end-to-end direct versus adjoint runtimes.

**Inputs:** No external project data required. Seeds and settings are in the code.

**Outputs:** `task9_validation_metrics.json`, `task9_validation_data.npz`, and `task9_validation_figure.png` in the notebook working folder.

**Important:** Timings vary with hardware and runtime conditions; run the experiment in the environment where timings will be reported.

In [2]:
"""Independent reproducibility extensions for the heat equation project.
Run: python task9_validation.py
"""
import json,time,sys
from pathlib import Path
import numpy as np
from scipy.optimize import minimize
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
P=Path.cwd()
alpha=.1; T=.1

def scalar_ftcs(N, steps):
    h=1/N; dt=T/steps; r=alpha*dt/h**2
    assert 4*r<=1+1e-12
    xx=np.linspace(0,1,N+1)[1:-1];X,Y=np.meshgrid(xx,xx,indexing='ij');S=np.sin(np.pi*X)*np.sin(np.pi*Y)
    u=np.zeros_like(S)
    for _ in range(steps):
        z=np.pad(u,1); u=u+r*(z[2:,1:-1]+z[:-2,1:-1]+z[1:-1,2:]+z[1:-1,:-2]-4*u)+dt*S
    lam_h=8*np.sin(np.pi/(2*N))**2/h**2
    a_h=-np.expm1(-alpha*lam_h*T)/(alpha*lam_h)
    a_exact=-np.expm1(-2*alpha*np.pi**2*T)/(2*alpha*np.pi**2)
    return np.linalg.norm(u-a_h*S)/np.linalg.norm(a_h*S), np.linalg.norm(u-a_exact*S)/np.linalg.norm(a_exact*S),r

spatial=[]
for N in [12,24,48]:
    # dt shrinks much faster than h^2; isolate spatial error relative to continuous exact solution
    steps=2000
    et,ec,r=scalar_ftcs(N,steps)
    lam_h=8*np.sin(np.pi/(2*N))**2*N**2
    ah=-np.expm1(-alpha*lam_h*T)/(alpha*lam_h)
    ae=-np.expm1(-2*alpha*np.pi**2*T)/(2*alpha*np.pi**2)
    spatial_exact=abs(ah-ae)/abs(ae)
    spatial.append(dict(N=N,steps=steps,dt=T/steps,temporal_vs_semidiscrete=et,total_vs_continuous=ec,spatial_vs_exact_semidiscrete=spatial_exact,stability=4*r))

temporal = []

for steps in [32, 64, 128, 256]:
    et, ec, r = scalar_ftcs(16, steps)
    temporal.append(dict(
        N=16,
        steps=steps,
        dt=T/steps,
        temporal_vs_semidiscrete=et,
        total_vs_continuous=ec,
        stability=4*r
    ))

def order(a, b, key, stepfield):
    return np.log(a[key]/b[key]) / np.log(a[stepfield]/b[stepfield])

spatial_order = [
    np.log(spatial[i]['spatial_vs_exact_semidiscrete'] /
           spatial[i+1]['spatial_vs_exact_semidiscrete']) / np.log(2)
    for i in range(2)
]

temporal_order = [
    np.log(temporal[i]['temporal_vs_semidiscrete'] /
           temporal[i+1]['temporal_vs_semidiscrete']) / np.log(2)
    for i in range(len(temporal)-1)
]

print('spatial', spatial, 'orders', spatial_order, flush=True)
print('temporal', temporal, 'orders', temporal_order, flush=True)


# Linear discrete forward and adjoint; all observations sampled on exact grid nodes.
N=24;nt=64;dt=T/nt;h=1/N
xx=np.linspace(0,1,N+1)[1:-1];X,Y=np.meshgrid(xx,xx,indexing='ij');ncell=X.size
obs_times=(16,32,64)
def basis(p):
    k=int(np.sqrt(p));assert k*k==p
    sigma= .11 if k==5 else .07
    centers=np.linspace(.12,.88,k)
    return np.stack([np.exp(-((X-a)**2+(Y-b)**2)/(2*sigma**2)) for a in centers for b in centers],axis=-1).reshape(ncell,p)
def step(u):
    z=np.pad(u,1)
    return u+alpha*dt*(z[2:,1:-1]+z[:-2,1:-1]+z[1:-1,2:]+z[1:-1,:-2]-4*u)/h**2

def field_observations(theta,B,nodes):
    u=np.zeros((N-1,N-1));F=(B@theta).reshape(u.shape); rows=[]
    for n in range(1,nt+1):
        u=step(u)+dt*F
        if n in obs_times: rows.append(u.ravel()[nodes].copy())
    return np.concatenate(rows)

def build_A(B,nodes):
    p=B.shape[1]; A=np.zeros((len(nodes)*len(obs_times),p))
    for j in range(p):
        theta=np.zeros(p);theta[j]=1.
        A[:,j]=field_observations(theta,B,nodes)
    return A

def ridge(A,d,rel=1e-3):
    m,p=A.shape;s=np.linalg.svd(A,compute_uv=False)
    reg=rel*(s[0]**2/m)
    return np.linalg.solve(A.T@A/m+reg*np.eye(p),A.T@d/m),reg,s

rng=np.random.default_rng(20261018)
B25=basis(25); truth=rng.uniform(.3,1.4,25)
allnodes=np.arange(ncell)
Aall=build_A(B25,allnodes)
# All candidate sensors are interior; repeated spatial layouts without replacement.
sensor_results={}
for nsensors in [6,16]:
    trials=[]
    for trial in range(90):
        chosen=np.sort(rng.choice(allnodes,size=nsensors,replace=False))
        indexes=np.concatenate([t*ncell+chosen for t in range(3)])
        A=Aall[indexes];clean=A@truth
        sd=.03*np.max(np.abs(clean))
        d=clean+rng.normal(0,sd,clean.size)
        sol,reg,s=ridge(A,d)
        err=np.linalg.norm(B25@(sol-truth))/np.linalg.norm(B25@truth)
        trials.append(dict(error=float(err),condition=float(s[0]/s[-1]) if s[-1]>0 else float('inf'),rank=int(np.linalg.matrix_rank(A)),lambda_value=float(reg)))
    errs=np.array([t['error'] for t in trials]);conds=np.array([t['condition'] for t in trials])
    sensor_results[str(nsensors)]=dict(trials=trials,mean_error=float(errs.mean()),median_error=float(np.median(errs)),p10=float(np.percentile(errs,10)),p90=float(np.percentile(errs,90)),median_condition=float(np.median(conds)),rank_min=min(t['rank'] for t in trials),rank_max=max(t['rank'] for t in trials))
    print('sensor',nsensors,'mean',errs.mean(),'median',np.median(errs),'p10-90',np.percentile(errs,[10,90]),'median cond',np.median(conds),flush=True)

# Complete work comparison: build A + direct least squares vs iterative adjoint PDE solve.
def adjoint_fg(theta,B,nodes,d,reg):
    y=field_observations(theta,B,nodes);m=len(d)
    residual=(y-d)/m
    injections={n:np.bincount(nodes,weights=residual[i*len(nodes):(i+1)*len(nodes)],minlength=ncell).reshape(N-1,N-1) for i,n in enumerate(obs_times)}
    q=np.zeros((N-1,N-1)); cumul=np.zeros_like(q)
    for n in range(nt,0,-1):
        if n in injections:q+=injections[n]
        cumul+=q;q=step(q)
    return .5*np.dot(y-d,y-d)/m+.5*reg*np.dot(theta,theta),dt*(B.T@cumul.ravel())+reg*theta

nodes=np.array([i*(N-1)+j for i in [3,8,13,18] for j in [3,8,13,18]])
bench={}
for p in [25,100]:
    B=basis(p);tr=np.random.default_rng(200+p).uniform(.3,1.3,p)
    clean=field_observations(tr,B,nodes)
    d=clean+np.random.default_rng(300+p).normal(0,.01*np.max(np.abs(clean)),len(clean))
    # Same objective, specified lambda, exact solve used as convergence reference.
    lambda_value=1e-7
    reps=[]
    for repeat in range(3):
        start=time.perf_counter();A=build_A(B,nodes)
        solution=np.linalg.solve(A.T@A/len(d)+lambda_value*np.eye(p),A.T@d/len(d))
        direct_elapsed=time.perf_counter()-start
        j_direct=.5*np.linalg.norm(A@solution-d)**2/len(d)+.5*lambda_value*np.linalg.norm(solution)**2
        start=time.perf_counter()
        opt=minimize(lambda th:adjoint_fg(th,B,nodes,d,lambda_value),np.zeros(p),method='L-BFGS-B',jac=True,options={'maxiter':800,'gtol':1e-12,'ftol':1e-15,'maxls':40})
        adj_elapsed=time.perf_counter()-start
        j_adj=adjoint_fg(opt.x,B,nodes,d,lambda_value)[0]
        relative_sol_err=np.linalg.norm(opt.x-solution)/np.linalg.norm(solution)
        reps.append(dict(matrix_total_s=direct_elapsed,adjoint_total_s=adj_elapsed,iterations=int(opt.nit),evaluations=int(opt.nfev),success=bool(opt.success),relative_coefficient_gap=float(relative_sol_err),objective_gap=float(j_adj-j_direct),gradient_norm=float(np.linalg.norm(adjoint_fg(opt.x,B,nodes,d,lambda_value)[1]))))
    bench[str(p)]=dict(repeats=reps,matrix_median_s=float(np.median([r['matrix_total_s'] for r in reps])),adjoint_median_s=float(np.median([r['adjoint_total_s'] for r in reps])),median_relative_coefficient_gap=float(np.median([r['relative_coefficient_gap'] for r in reps])))
    print('bench',p,bench[str(p)]['matrix_median_s'],bench[str(p)]['adjoint_median_s'],bench[str(p)]['median_relative_coefficient_gap'],flush=True)

results=dict(experiment='Task 9: three validation extensions',numpy_version=np.__version__,seed=20261018,spatial_convergence=dict(rows=spatial,observed_orders=spatial_order),temporal_convergence=dict(rows=temporal,observed_orders=temporal_order),sensor_placement=sensor_results,end_to_end=bench,definitions=dict(sensor_error='spatial discrete relative L2 norm of B(theta_est-theta_true) divided by norm(B theta_true)',sensor_noise='Gaussian standard deviation 3% of maximum clean observation for each sensor layout',condition='sigma_max/sigma_min on positive singular values; rectangular condition for sparse configurations',regularization='lambda=10^-3 sigma_max(A)^2/m for random sensor experiments; lambda=10^-7 for timing comparison',timing='median of three sequential wall-clock measurements on this runtime; matrices rebuilt every repetition',spatial='pure spatial error: exact semi-discrete heat-equation solution versus exact continuum solution at the same interior grid nodes; total FTCS error also reported independently',temporal='time-stepping error vs exact semi-discrete sinusoidal amplitude for fixed spatial grid'))
(P/'task9_validation_metrics.json').write_text(json.dumps(results,indent=2))
np.savez_compressed(P/'task9_validation_data.npz',spatial_N=np.array([r['N'] for r in spatial]),spatial_error=np.array([r['spatial_vs_exact_semidiscrete'] for r in spatial]),temporal_steps=np.array([r['steps'] for r in temporal]),temporal_error=np.array([r['temporal_vs_semidiscrete'] for r in temporal]),sensor6=np.array([r['error'] for r in sensor_results['6']['trials']]),sensor16=np.array([r['error'] for r in sensor_results['16']['trials']]))
fig,axs=plt.subplots(2,2,figsize=(12,9))
axs[0,0].loglog([1/r['N'] for r in spatial],[r['spatial_vs_exact_semidiscrete'] for r in spatial],'o-');axs[0,0].set(xlabel='Grid spacing h',ylabel='Relative error',title='Spatial refinement: semi-discrete exact');axs[0,0].grid(alpha=.25)
axs[0,1].loglog([r['dt'] for r in temporal],[r['temporal_vs_semidiscrete'] for r in temporal],'o-');axs[0,1].set(xlabel='Time step Δt',ylabel='Relative error',title='Temporal refinement: semi-discrete reference');axs[0,1].grid(alpha=.25)
axs[1,0].boxplot([[100*r['error'] for r in sensor_results[s]['trials']] for s in ['6','16']],tick_labels=['6 sensors','16 sensors'],showfliers=False);axs[1,0].set(ylabel='Relative source error (%)',title='90 randomized sensor layouts per configuration')
for idx,p in enumerate(['25','100']):
    x=np.array([0,1])+idx*2.7
    axs[1,1].bar(x,[bench[p]['matrix_median_s'],bench[p]['adjoint_median_s']],width=.7)
axs[1,1].set_xticks([0,1,2.7,3.7],['25 matrix','25 adjoint','100 matrix','100 adjoint'],rotation=20);axs[1,1].set(ylabel='Median full-solve time (s)',title='End-to-end timings (same objective)')
fig.tight_layout();fig.savefig(P/'task9_validation_figure.png',dpi=165);plt.close(fig)
print('DONE',flush=True)


spatial [{'N': 12, 'steps': 2000, 'dt': 5e-05, 'temporal_vs_semidiscrete': np.float64(4.4411804953389757e-05), 'total_vs_continuous': np.float64(0.0005885693048282125), 'spatial_vs_exact_semidiscrete': np.float64(0.0005441333339313212), 'stability': 0.00288}, {'N': 24, 'steps': 2000, 'dt': 5e-05, 'temporal_vs_semidiscrete': np.float64(4.458319113633715e-05), 'total_vs_continuous': np.float64(0.00018081820455795737), 'spatial_vs_exact_semidiscrete': np.float64(0.000136228939900748), 'stability': 0.01152}, {'N': 48, 'steps': 2000, 'dt': 5e-05, 'temporal_vs_semidiscrete': np.float64(4.462610447328949e-05), 'total_vs_continuous': np.float64(7.869709249833384e-05), 'spatial_vs_exact_semidiscrete': np.float64(3.406946763742341e-05), 'stability': 0.04608}] orders [np.float64(1.997926995509564), np.float64(1.9994819064872302)]
temporal [{'N': 16, 'steps': 32, 'dt': 0.003125, 'temporal_vs_semidiscrete': np.float64(0.0027924217696787402), 'total_vs_continuous': np.float64(0.003099608859846666), 